In [1]:
import pandas as pd
import numpy as np
df = pd.read_parquet("../data/processed/sepsis_raw_combined.parquet")
print(df.shape)

(1552210, 42)


In [2]:
# List of vital signs and lab test columns (excluding demographics/time columns)
vitals_and_labs = ["HR","O2Sat","Temp","SBP","MAP","DBP","Resp","EtCO2",
                    "BaseExcess","HCO3","FiO2","pH","PaCO2","SaO2",
                    "AST","BUN","Alkalinephos","Calcium","Chloride",
                    "Creatinine","Bilirubin_direct","Glucose","Lactate",
                    "Magnesium","Phosphate","Potassium","Bilirubin_total",
                    "TroponinI","Hct","Hgb","PTT","WBC","Fibrinogen","Platelets"]

# Calculate missing percentage for each of these columns
missing_pct = df[vitals_and_labs].isna().mean() * 100

# Tier 1: columns with more than 95% missing values -> will only get a binary flag (measured or not)
tier1_high = missing_pct[missing_pct > 95].index.tolist()

# Tier 2: columns with 35% to 95% missing values -> will be filled using MICE (smart imputation)
tier2_mid = missing_pct[(missing_pct > 35) & (missing_pct <= 95)].index.tolist()

# Tier 3: columns with less than 35% missing values -> will be filled using forward/backward fill
tier3_low = missing_pct[missing_pct <= 35].index.tolist()

# Print how many columns fall into each tier
print("Tier 1 (>95% missing):", len(tier1_high), "columns")
print("Tier 2 (35-95% missing):", len(tier2_mid), "columns")
print("Tier 3 (<35% missing):", len(tier3_low), "columns")

Tier 1 (>95% missing): 13 columns
Tier 2 (35-95% missing): 15 columns
Tier 3 (<35% missing): 6 columns


In [6]:
# Print the actual column names in each tier to see which features fall where
print("Tier 1 (only binary flag):", tier1_high)
print()
print("Tier 2 (MICE imputation):", tier2_mid)
print()
print("Tier 3 (forward/backward fill):", tier3_low)

Tier 1 (only binary flag): ['EtCO2', 'HCO3', 'SaO2', 'AST', 'Alkalinephos', 'Chloride', 'Bilirubin_direct', 'Lactate', 'Phosphate', 'Bilirubin_total', 'TroponinI', 'PTT', 'Fibrinogen']

Tier 2 (MICE imputation): ['Temp', 'BaseExcess', 'FiO2', 'pH', 'PaCO2', 'BUN', 'Calcium', 'Creatinine', 'Glucose', 'Magnesium', 'Potassium', 'Hct', 'Hgb', 'WBC', 'Platelets']

Tier 3 (forward/backward fill): ['HR', 'O2Sat', 'SBP', 'MAP', 'DBP', 'Resp']


In [7]:
# For each Tier 1 column, create a new binary flag column
# 1 = this test WAS measured for this patient at this hour, 0 = it was NOT measured
for c in tier1_high:
    df[f"{c}_measured"] = df[c].notna().astype(int)

# Now drop the original Tier 1 columns since their actual values are too unreliable (>95% missing)
df = df.drop(columns=tier1_high)

# Print new shape to confirm
print(df.shape)

(1552210, 42)


In [8]:
# Print all column names to see the change
print(df.columns.tolist())

['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'BaseExcess', 'FiO2', 'pH', 'PaCO2', 'BUN', 'Calcium', 'Creatinine', 'Glucose', 'Magnesium', 'Potassium', 'Hct', 'Hgb', 'WBC', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel', 'patient_id', 'EtCO2_measured', 'HCO3_measured', 'SaO2_measured', 'AST_measured', 'Alkalinephos_measured', 'Chloride_measured', 'Bilirubin_direct_measured', 'Lactate_measured', 'Phosphate_measured', 'Bilirubin_total_measured', 'TroponinI_measured', 'PTT_measured', 'Fibrinogen_measured']


In [9]:
# For each Tier 3 column (vitals with low missingness), fill missing values
for c in tier3_low:
    # Forward fill: carry the last known value forward within each patient
    df[c] = df.groupby("patient_id")[c].ffill()
    # Backward fill: for any remaining missing values at the START of a patient's stay,
    # use the next known value backward (in case the patient's very first hours had no reading)
    df[c] = df.groupby("patient_id")[c].bfill()

# Check how much missing is left in these columns
print(df[tier3_low].isna().sum())

HR          142
O2Sat       374
SBP       10976
MAP        2698
DBP      259250
Resp       1999
dtype: int64


In [10]:
# For any remaining missing values in Tier 3 columns (entire patient stay had no reading),
# fill with the overall median value of that column across all patients
for c in tier3_low:
    df[c] = df[c].fillna(df[c].median())

# Confirm no missing values remain in Tier 3 columns
print(df[tier3_low].isna().sum())

HR       0
O2Sat    0
SBP      0
MAP      0
DBP      0
Resp     0
dtype: int64


In [11]:
# Import required tools for MICE imputation from scikit-learn
from sklearn.experimental import enable_iterative_imputer  # this line is required to "unlock" IterativeImputer
from sklearn.impute import IterativeImputer

print("MICE tools imported successfully")

MICE tools imported successfully


In [12]:
# Take a random sample of 50,000 rows to quickly test if MICE works before running on full data
sample_df = df[tier2_mid].sample(50000, random_state=42)

# Create the MICE imputer object
# max_iter=5 means it will refine its guesses 5 times for better accuracy
imputer = IterativeImputer(max_iter=5, random_state=42, sample_posterior=True)

# Fit the imputer on the sample and transform it (fill missing values)
imputed_sample = imputer.fit_transform(sample_df)

# Print the shape to confirm it worked
print("MICE sample test done, shape:", imputed_sample.shape)

MICE sample test done, shape: (50000, 15)


In [13]:
# Import time module to measure how long this takes
import time

# Record the start time
start = time.time()

# Create a fresh imputer for the full dataset
# max_iter=3 (fewer iterations than sample test, to keep runtime reasonable on full data)
imputer_full = IterativeImputer(max_iter=3, random_state=42, sample_posterior=True)

# Fit and transform on ALL Tier 2 columns across the entire dataset (1.5 million rows)
imputed_full = imputer_full.fit_transform(df[tier2_mid])

# Print how many minutes this took
print("Time taken:", (time.time() - start) / 60, "minutes")

# Print the shape to confirm all rows were processed
print("Shape:", imputed_full.shape)

Time taken: 0.9803699413935344 minutes
Shape: (1552210, 15)


In [14]:
# Replace the original Tier 2 columns (with missing values) with the MICE-imputed values
df[tier2_mid] = imputed_full

# Confirm all Tier 2 columns now have zero missing values
print(df[tier2_mid].isna().sum())

Temp          0
BaseExcess    0
FiO2          0
pH            0
PaCO2         0
BUN           0
Calcium       0
Creatinine    0
Glucose       0
Magnesium     0
Potassium     0
Hct           0
Hgb           0
WBC           0
Platelets     0
dtype: int64


In [15]:
# Check total missing values across the ENTIRE dataframe (not just one tier)
print(df.isna().sum().sum(), "total missing values")

# If any columns still have missing values, show which ones
print(df.isna().sum()[df.isna().sum() > 0])

1223928 total missing values
Unit1          611960
Unit2          611960
HospAdmTime         8
dtype: int64


In [16]:
# Unit1, Unit2 have a lot of missing values - fill with 0 (meaning "unknown ICU type")
df["Unit1"] = df["Unit1"].fillna(0)
df["Unit2"] = df["Unit2"].fillna(0)

# HospAdmTime has only 8 missing values - fill with median (safe simple fix for very few missing values)
df["HospAdmTime"] = df["HospAdmTime"].fillna(df["HospAdmTime"].median())

# Final check: confirm ENTIRE dataframe now has zero missing values
print(df.isna().sum().sum(), "total missing values (should be 0)")

0 total missing values (should be 0)


In [17]:
# Define physiologically plausible ranges for key vital signs
# Any value outside these ranges is likely a sensor error or data entry mistake
bounds = {
    "HR": (0, 300),      # Heart rate: 0 to 300 beats per minute
    "O2Sat": (0, 100),   # Oxygen saturation: 0 to 100 percent
    "Temp": (25, 45),    # Temperature: 25 to 45 degrees Celsius
    "SBP": (0, 300),     # Systolic blood pressure: 0 to 300 mmHg
    "MAP": (0, 250),     # Mean arterial pressure: 0 to 250 mmHg
    "DBP": (0, 200),     # Diastolic blood pressure: 0 to 200 mmHg
    "Resp": (0, 80),     # Respiratory rate: 0 to 80 breaths per minute
}

# Clip (limit) each column's values to stay within its physiologically valid range
for col, (low, high) in bounds.items():
    df[col] = df[col].clip(low, high)

print("outlier bounds applied")

outlier bounds applied


In [18]:
# Check total missing values across the ENTIRE dataframe (not just one tier)
print(df.isna().sum().sum(), "total missing values")

# If any columns still have missing values, show which ones
print(df.isna().sum()[df.isna().sum() > 0])

0 total missing values
Series([], dtype: int64)


In [19]:
# Save the fully cleaned dataset (with advanced tiered missing-data handling) to disk
df.to_parquet("../data/processed/sepsis_cleaned_final.parquet")

print("saved cleaned dataset!")

saved cleaned dataset!
